# CuTeDSL 入门：布局（Layout）、`from_dlpack` 与最小向量加法（H100 / Hopper）

**定位**：在 **NVIDIA H100（Hopper，SM 9.0）** 上，用 [CUTLASS Python DSL / CuTeDSL](https://github.com/NVIDIA/cutlass)（PyPI 包名常见为 **`nvidia-cutlass-dsl`**）完成两件事：（1）理解 CuTe 里「张量 = 存储 + 布局」；（2）用 **`cutlass.cute.runtime.from_dlpack`** 把 PyTorch GPU 张量接进 DSL，并用 **`@cute.jit`** 写一个整向量级别的加法 Demo。

**第三方许可**：CuTeDSL / CUTLASS 版权归 NVIDIA，使用受 [Python DSL 许可证说明](https://docs.nvidia.com/cutlass/media/docs/pythonDSL/license.html) 约束。本 Notebook 的讲解顺序、注释与实验步骤为原创编排；API 用法请参考官方文档与仓库示例。

**可分发说明**：可用于学习、教学或发布到 Gitee 等平台；请同时遵守 NVIDIA EULA 与各依赖包许可证。

---

## 硬件与软件（本教程默认仅 Hopper）

| 项目 | 说明 |
|------|------|
| **GPU** | **仅支持 NVIDIA H100（compute capability 9.0）**。下文自检若 `major != 9` 会直接退出，避免在错误架构上误跑。 |
| **显存** | 本 Demo 仅小向量；**≥ 4GB** 即可，与 H100 标配相比非常轻。 |
| **CUDA / 驱动** | 与所安装的 **`nvidia-cutlass-dsl` / `cuda-python`** 组合一致；驱动需支持你的 CUDA 工具链。 |
| **Python** | 语法按 **3.8+** 编写；**实际下限以 `nvidia-cutlass-dsl` 轮子要求为准**。 |
| **禁止** | 本课**不**使用 Blackwell（SM10.x）专属 API；也不以 `cutlass/test/examples/CuTeDSL/sm_100a/` 为主模板。 |

---

## 依赖安装（在终端执行，建议虚拟环境）

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip

# PyTorch（CUDA 版本请按 https://pytorch.org 与你本机驱动选择，以下为 cu124 索引示例）
pip install torch --index-url https://download.pytorch.org/whl/cu124

# CuTeDSL（以 PyPI 包为准；若官方文档包名有调整，以文档为准）
pip install nvidia-cutlass-dsl

# 可选：从本仓库可编辑安装（需匹配 CUTLASS_PATH、CUDA、cuda-python 版本）
# export CUTLASS_PATH=/path/to/cutlass
# cd "$CUTLASS_PATH/python/CuTeDSL" && ./setup.sh --cu12   # 或 --cu13，见该目录说明
# pip install -e .
# 运行本 Notebook 需要 Jupyter
pip install jupyter
```

更多背景见 CUTLASS 文档：[install.md](https://github.com/NVIDIA/cutlass/blob/main/python/docs_src/source/install.md)（路径以你克隆的仓库为准）。

---

## 如何打开本 Notebook

在仓库根目录（或包含 本notebook 的路径）执行：

```bash
jupyter-lab --ip=0.0.0.0 --port=8888 --no-browser --allow-root
```

---

In [1]:
# 环境自检：Python、CUDA、Hopper、cutlass.cute
import sys

print("Python:", sys.version)

import torch

if not torch.cuda.is_available():
    print("错误：未检测到 CUDA。请在带 NVIDIA GPU 且已正确安装 PyTorch CUDA 版的环境中运行。")
    sys.exit(1)

major, minor = torch.cuda.get_device_capability(0)
print(f"GPU0 capability: ({major}, {minor})")
if major != 9:
    print(
        "错误：本教程仅针对 Hopper / H100（major == 9）。"
        "当前 GPU 不满足，已停止，避免在错误架构上误导。"
    )
    sys.exit(1)

import cutlass.cute as cute
from cutlass.cute import Tensor
from cutlass.cute.runtime import from_dlpack

print("cutlass.cute 导入成功。")

Python: 3.12.3 (main, Feb  4 2025, 14:48:35) [GCC 13.3.0]
GPU0 capability: (9, 0)
cutlass.cute 导入成功。


## 本课概念（3 分钟版）

- **Layout**：把逻辑坐标映射到「扁平」内存下标。`cutlass.cute.make_layout(shape, stride=...)` 在 **内核 / JIT 代码**里常用；不传 `stride` 时由形状推紧凑步长（与 C++ CuTe 一致，详见 `cutlass/python/CuTeDSL/cutlass/cute/core.py` 文档字符串）。
- **Tensor（CuTe 意义）**：可理解为 **指针（或引擎）+ 布局**，访问坐标时先算偏移再读写。运行时侧可用 **`from_dlpack`** 把 **支持 DLPack 的张量**（如 `torch.Tensor`）包成 DSL 可见的 `Tensor`，并读取其 `shape` / `stride` / `layout`。
- **本 Demo 的算子**：对 **一维、长度一致** 的三个 `float32` GPU 张量，用 **`@cute.jit`** 声明的函数做一次 **整段向量** 的 `out = a + b`（与 `cutlass.cute.typing` 中 `Tensor` 文档示例同型，只是把数据放在 **CUDA** 上以便在 H100 上执行）。

---

In [2]:
# 用 PyTorch 在 GPU 上准备一维张量，并通过 DLPack 观察 CuTe 侧的 shape / stride / layout
n = 256
torch.manual_seed(0)
a = torch.randn(n, device="cuda", dtype=torch.float32)
b = torch.randn(n, device="cuda", dtype=torch.float32)

t_a = from_dlpack(a)
print("shape:", t_a.shape)
print("stride:", t_a.stride)

shape: (256,)
stride: (1,)


### 为何要用 `@cute.jit`？

函数里的 `load` / `store` 是 **CuTeDSL 原语**，会走 **MLIR → GPU 代码** 的编译，而不是在 Python 里像普通变量那样读写内存。**`@cute.jit`** 表示「这是 DSL 函数」：首次调用时会按实参张量的类型与布局 **JIT 编译** 再执行。没有它，这段代码既不能当普通 Python 跑，也通常无法生成你想要的设备端向量加。


In [3]:
# 最小 @cute.jit：整向量 load / store（与 cutlass.cute.typing.Tensor 文档中的 add 示例同型）
# 默认用 CPU 张量：若干环境下对 GPU 整段 gmem 的 load/store 可能在 JIT 编译或执行时段错误，
# Jupyter 会表现为 “The kernel appears to have died”（无 Python traceback）。
@cute.jit
def vec_add_f32(x: Tensor, y: Tensor, out: Tensor):
    out.store(x.load() + y.load())


# --- A) 推荐：CPU float32 小向量（稳定、与官方文档路径一致）---
a_cpu = torch.tensor([1.0, 2.0, 3.0], dtype=torch.float32)
b_cpu = torch.tensor([4.0, 5.0, 6.0], dtype=torch.float32)
out_cpu = torch.zeros_like(a_cpu)
vec_add_f32(from_dlpack(a_cpu), from_dlpack(b_cpu), from_dlpack(out_cpu))
ref_cpu = a_cpu + b_cpu
print("CPU out:", out_cpu.tolist(), " allclose:", torch.allclose(out_cpu, ref_cpu))

CPU out: [5.0, 7.0, 9.0]  allclose: True


In [ ]:
# --- B) 可选：在 GPU 上再跑同一语义（仅当你的 nvidia-cutlass-dsl + 驱动组合稳定时启用）---
RUN_GPU_VEC_ADD = True  # 若你确认不会 kernel 崩溃，可改为 True
if RUN_GPU_VEC_ADD:
    out_g = torch.empty_like(a)
    vec_add_f32(from_dlpack(a), from_dlpack(b), from_dlpack(out_g))
    torch.cuda.synchronize()
    ok_g = torch.allclose(out_g, a + b)
    print("GPU allclose(out, a+b):", ok_g)
    if not ok_g:
        raise RuntimeError("GPU 数值校验失败。")

## 常见报错（节选）

1. **`ModuleNotFoundError: cutlass` / `_mlir`**：未安装 **`nvidia-cutlass-dsl`** 或安装损坏；按上文 `pip install` 重装，或改用可编辑安装并保证 `CUTLASS_PATH`、CUDA、`cuda-python` 版本一致。
2. **自检提示「仅支持 Hopper」**：当前 GPU 不是 SM 9.0（例如 A100、消费级卡）。本文件有意 **不** 在非 9.x 上继续跑，以免与 Hopper 专用示例混淆。
3. **`nvcc` / PTX 相关版本错误**：DSL 编译链与本地 CUDA 不匹配；对齐 `nvcc --version`、驱动与 `cuda-python` 所期望的 CUDA major/minor。

---